# Week 1 · Day 5 — Multi-step Brochure Generator

So far we've made *single* LLM calls. This notebook chains several together into
a small pipeline that turns a company's website into a marketing brochure:

1. **Scrape** the landing page and collect every link on it.
2. **Ask the model** which of those links matter for a brochure (About, Careers,
   Products — not Terms, Privacy, or social icons).
3. **Gather** the page text plus those relevant links.
4. **Ask the model again** to write the brochure from everything we collected.

The key idea: the LLM is used both as a **filter** (step 2) and a **generator**
(step 4), with the output of one call feeding the input of the next.

> **Prerequisite:** `OPENAI_API_KEY` in your `.env`. Scraping helpers come from
> [scraper.py](scraper.py).

## 1. Setup

In [12]:
import json
import os

from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI
from scraper import fetch_website_content, fetch_website_links

In [13]:
load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
MODEL = "gpt-5.4-mini"

## 2. Step one — let the model choose the relevant links

A landing page links to dozens of places: nav, footer, legal, social. Most are
useless for a brochure. Rather than hand-writing filter rules, we hand the whole
list to the model and let it decide.

- `get_links_user_prompt` builds the instruction, embedding the **link list**
  (from `fetch_website_links(..., extract=True)`).
- `select_relevant_links` makes the call. `response_format={"type": "json_object"}`
  forces valid JSON back, which we parse into a plain Python list.

In [14]:
def get_links_user_prompt(url: str, links: list[str]) -> str:
    return f"""Here's a list of links found on {url}. Decide which are relevant
for designing a brochure about the website — keep things like About, Products,
Careers, or news sections, and drop Terms, Privacy, login, ad-choices, and
social-share links.

Respond with a JSON object of exactly this shape, using absolute URLs:
{{"relevant_links": ["https://full.url/one", "https://full.url/two"]}}

Here are the links:
{links}"""

In [15]:
def select_relevant_links(url: str, links: list[str]) -> list[str]:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": "You select the relevant links for designing a brochure about a website. Respond in JSON.",
            },
            {"role": "user", "content": get_links_user_prompt(url=url, links=links)},
        ],
        response_format={"type": "json_object"},
    )
    data = json.loads(response.choices[0].message.content)  # type: ignore
    return data.get("relevant_links", [])

Quick check — fetch CNN's links and see what the model keeps.

> `extract=True` returns the **list of links** (a few KB). `extract=False` would
> return the raw HTML (~1M tokens for CNN) and blow past the model's per-minute
> token limit — that's the 429 trap to avoid here.

In [16]:
url = "https://www.cnn.com"
select_relevant_links(url=url, links=fetch_website_links(url=url, extract=True))

['https://edition.cnn.com',
 'https://edition.cnn.com/us',
 'https://edition.cnn.com/world',
 'https://edition.cnn.com/politics',
 'https://edition.cnn.com/business',
 'https://edition.cnn.com/health',
 'https://edition.cnn.com/entertainment',
 'https://edition.cnn.com/style',
 'https://edition.cnn.com/travel',
 'https://edition.cnn.com/sports',
 'https://edition.cnn.com/science',
 'https://edition.cnn.com/climate',
 'https://edition.cnn.com/weather',
 'https://edition.cnn.com/sport/fifa-world-cup',
 'https://edition.cnn.com/world/europe/ukraine',
 'https://edition.cnn.com/world/middleeast/israel',
 'https://edition.cnn.com/games',
 'https://edition.cnn.com/watch',
 'https://edition.cnn.com/audio',
 'https://edition.cnn.com/live-tv',
 'https://www.cnn.com/newsletters',
 'https://us.cnn.com?hpt=header_edition-picker',
 'https://edition.cnn.com?hpt=header_edition-picker',
 'https://arabic.cnn.com?hpt=header_edition-picker',
 'https://cnnespanol.cnn.com/?hpt=header_edition-picker',
 'http

## 3. Gather the page content and relevant links

`fetch_page_and_relevant_links` bundles step one with a scrape of the landing
page, returning everything the brochure step needs in a single dict.

In [ ]:
def fetch_page_and_relevant_links(url: str) -> dict:
    content = fetch_website_content(url=url)
    links = fetch_website_links(url=url, extract=True)   # list of links, not raw HTML
    relevant_links = select_relevant_links(url=url, links=links)
    return {"content": content, "relevant_links": relevant_links}

## 4. Step two — generate the brochure

The second LLM call. The **system prompt** defines the brochure structure; the
**user prompt** carries the data we gathered in step 3.

In [18]:
brochure_system_prompt = """You are a helpful assistant that designs brochures for websites. You will be given the content of a website and a list of relevant links. Use this information to design a brochure for the website. The brochure should include the following sections:
1. Overview: A brief summary of the website and its purpose.
2. Key Features: A list of the key features of the website, with a brief description.
3. Target Audience: A description of the target audience for the website.
4. Relevant Links: A list of the relevant links, with a brief description of each.
Make the brochure visually appealing and easy to read. Use headings, bullet points, and other markdown formatting."""

In [19]:
def create_brochure(url: str) -> str:
    page = fetch_page_and_relevant_links(url=url)

    # Trim the page text so a content-heavy site stays under the per-minute
    # token limit (the brochure quality barely changes past the first ~20k chars).
    content = page["content"][:20_000]

    user_prompt = f"""Here's the content of the website:
{content}

And here are some relevant links:
{page['relevant_links']}

Please design a brochure for the website based on this information."""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    )
    return response.choices[0].message.content  # type: ignore

Run the whole pipeline end-to-end and render the brochure as markdown.

In [20]:
brochure = create_brochure(url="https://www.cnn.com")
display(Markdown(brochure))

# CNN — Breaking News, Latest News and Videos

## Overview
CNN is a global news website delivering **breaking news, live updates, analysis, and video coverage** across major categories like world affairs, U.S. politics, business, health, entertainment, sports, science, climate, and more. It serves as a comprehensive news hub for readers who want timely reporting, in-depth stories, and multimedia content from around the world.

---

## Key Features

- **Breaking News & Live Updates**
  - Real-time coverage of major events, crises, elections, and global developments.

- **Wide Range of News Sections**
  - Dedicated coverage for **U.S., World, Politics, Business, Health, Entertainment, Style, Travel, Sports, Science, Climate,** and **Weather**.

- **Video and Live TV**
  - Extensive video library, clips, and live broadcasts for users who prefer visual news coverage.

- **Topic-Based Navigation**
  - Follow specific subjects like **Ukraine-Russia War, Israel-Hamas War, World Cup 2026, Epstein Files,** and more.

- **Regional & International Editions**
  - Content tailored for different audiences, including **U.S., International, Arabic,** and **Español** editions.

- **Opinion, Analysis, and Features**
  - Includes analysis pieces, explainers, investigative reporting, and long-form features.

- **Podcasts and Audio**
  - Audio storytelling and podcast programming for on-the-go listening.

- **Interactive Tools & Games**
  - Crossword, Sudoku, quizzes, and other interactive content for audience engagement.

- **Mobile App Access**
  - Promotes downloading the CNN app for easier access to news on mobile devices.

- **Account & Newsletter Options**
  - Users can sign in, manage preferences, and subscribe to newsletters/topics they follow.

---

## Target Audience

CNN is designed for:

- **News readers** seeking up-to-the-minute headlines and reliable reporting
- **Professionals and analysts** who need business, political, and global news
- **International audiences** looking for regional and translated editions
- **Sports, entertainment, and lifestyle readers** interested in topic-specific coverage
- **Mobile and video-first users** who prefer watchable news content and live streams
- **Engaged audiences** who want newsletters, podcasts, and curated topic tracking

---

## Relevant Links

### Main Site & Editions
- [CNN Home](https://edition.cnn.com) — Main homepage for the latest headlines and featured stories.
- [U.S. Edition](https://edition.cnn.com?hpt=header_edition-picker) — U.S.-focused CNN edition.
- [International Edition](https://edition.cnn.com?hpt=header_edition-picker) — Global edition with broader international coverage.
- [Arabic Edition](https://arabic.cnn.com?hpt=header_edition-picker) — CNN content in Arabic.
- [Español Edition](https://cnnespanol.cnn.com/?hpt=header_edition-picker) — CNN content in Spanish.

### Core News Sections
- [U.S.](https://edition.cnn.com/us) — American national news and major domestic stories.
- [World](https://edition.cnn.com/world) — International news coverage from around the globe.
- [Politics](https://edition.cnn.com/politics) — U.S. political reporting, elections, and government.
- [Business](https://edition.cnn.com/business) — Markets, companies, finance, and economic news.
- [Health](https://edition.cnn.com/health) — Health news, wellness, and medical reporting.
- [Entertainment](https://edition.cnn.com/entertainment) — Movies, TV, celebrities, and pop culture.
- [Style](https://edition.cnn.com/style) — Fashion, design, luxury, and cultural trends.
- [Travel](https://edition.cnn.com/travel) — Destinations, food, and travel features.
- [Sports](https://edition.cnn.com/sports) — General sports coverage and highlights.
- [Science](https://edition.cnn.com/science) — Science, discovery, and research stories.
- [Climate](https://edition.cnn.com/climate) — Climate reporting and environmental solutions.
- [Weather](https://edition.cnn.com/weather) — Weather forecasts, alerts, and weather videos.

### Special Coverage
- [World Cup 2026](https://edition.cnn.com/sport/fifa-world-cup) — Coverage, guides, and stories on the tournament.
- [Ukraine-Russia War](https://edition.cnn.com/world/ukraine) — Ongoing war coverage and analysis.
- [Israel-Hamas War](https://edition.cnn.com/world/middleeast/israel) — Coverage of conflict and regional developments.

### Watch, Listen, and Live
- [Watch](https://edition.cnn.com/watch) — Video hub for CNN clips and programs.
- [Audio](https://edition.cnn.com/audio) — Podcasts and audio content.
- [Live TV](https://edition.cnn.com/live-tv) — Stream CNN live.

### Games & Interactive
- [Games](https://edition.cnn.com/games) — CNN’s games and puzzles section.
- [CNN Crossword](https://edition.cnn.com/games/play/cnn-crossword) — Daily crossword puzzle.
- [Jumble Crossword](https://edition.cnn.com/games/play/jumble-crossword-daily) — Word puzzle challenge.
- [Sudoblock](https://edition.cnn.com/games/play/sudoblock) — Block-based puzzle game.
- [Daily Sudoku](https://edition.cnn.com/games/play/daily-sudoku) — Sudoku puzzles for daily play.
- [5 Things Quiz](https://cnn.it/5thingsquiz) — Quick quiz tied to CNN’s “5 Things” content.

### Newsletters & Personalization
- [Newsletters](https://www.cnn.com/newsletters) — Sign up for curated email updates and newsletters.

### About & Company Info
- [About CNN](https://edition.cnn.com/about) — Company overview and background.
- [CNN Profiles](https://edition.cnn.com/profiles) — Reporter and contributor profiles.
- [CNN Leadership](https://edition.cnn.com/profiles/cnn-leadership) — Leadership information.
- [Accessibility](https://edition.cnn.com/accessibility) — Accessibility support and information.
- [Transcripts](https://edition.cnn.com/transcripts) — Video and broadcast transcripts.
- [Careers](https://careers.wbd.com/cnnjobs) — Job opportunities at CNN.

---

If you'd like, I can also turn this into a **one-page marketing brochure layout** or a **tri-fold brochure format** with a more polished visual structure.